## Object Detection in MS COCO

In [ ]:
import os
import torch
import numpy as np
from torchvision import transforms
#--------------------------------------------
from skimage.filters import gaussian
from scipy.ndimage import gaussian_filter
import cv2

import matplotlib.pyplot as plt

import shap_bpt
print('shap_bpt version:',shap_bpt.__version__)

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if ('mps' in dir(torch.backends)) and torch.backends.mps.is_available() else torch.device("cpu")
device

In [ ]:
import os
from pathlib import Path
import yaml

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'shap_bpt').is_dir():
            return path
    raise FileNotFoundError('Could not find the project root from the current working directory.')

original_working_dir = Path.cwd()
project_root = find_project_root(original_working_dir)
os.chdir(project_root)
print(f'Project root: {project_root}')

# config_file = "MSCOCO_mac"
config_file = "MSCOCO_xn2"

try:
    with open(project_root / f"examples/configs/{config_file}.yaml", "r") as f:
        config = yaml.safe_load(f)
finally:
    os.chdir(original_working_dir)
    print(f'Restored working directory: {original_working_dir}')

dataset_root = config["data"]["dataset_root"]
print(f"Dataset root: {dataset_root}")

In [ ]:
from ultralytics import YOLO

model = YOLO(f'{original_working_dir}/checkpoints/yolo11s.pt')
class_names = model.names

print(f'{"Num_Classes":<15}{len(class_names)}')

model.info()

from pycocotools.coco import COCO

model_preprocess = transforms.Compose(
    [transforms.ToTensor()]
)

In [ ]:
def load_image(fname,im_size=None,bg_type='black'):
    img_ = cv2.imread(f'{fname}')
    image_to_explain = cv2.cvtColor(img_, cv2.COLOR_BGR2RGB)                #.astype(np.float32)
    if im_size is not None:
        image_to_explain         = cv2.resize(image_to_explain,im_size)     # [:,:,::-1]
    image_to_explain_preproc  = image_to_explain.copy()                     #torch.tensor(image_to_explain).to(device)# .astype(np.float32)/255.0
    np.random.seed(0)
    bkgnd0 = np.full_like(image_to_explain, 0)
    bkgnd1 = np.full_like(image_to_explain, 127)
    bkgnd2 = np.full_like(image_to_explain, 255)
    bkgnd3 = gaussian(image_to_explain, 8, channel_axis=-1)*255
    bkgnd4 = np.clip(np.random.normal(128, 128, size=image_to_explain.shape), 0, 255).astype(np.uint8)
    bkgnd4 = (gaussian(bkgnd4, 2.0, channel_axis=-1) * 255).astype(np.uint8)
    if bg_type=='black': background_image_set = np.array([bkgnd0])
    elif bg_type=='gray': background_image_set = np.array([bkgnd1])
    elif bg_type=='white': background_image_set = np.array([bkgnd2])
    elif bg_type=='blurred': background_image_set = np.array([bkgnd3])
    elif bg_type=='noise': background_image_set = np.array([bkgnd4])
    elif bg_type=='full': background_image_set = np.array([bkgnd0, bkgnd1, bkgnd2, bkgnd3, bkgnd4])
    else: raise ValueError(f'Unknown bg_type: {bg_type}')
    background_image_preproc_set = [model_preprocess(bkgnd.astype(np.float32)/255.0)
                                        for bkgnd in background_image_set]
    background_tensors = torch.cat([torch.unsqueeze(bk_p, dim=0) 
                                    for bk_p in background_image_preproc_set]).to(device)
    return image_to_explain,image_to_explain_preproc,background_image_set,background_tensors

In [ ]:
def predict_yolo(x,coco_classes_count=80,verbose=False):
    res = model.predict(source=x, verbose=verbose)[0]
    p = np.zeros(coco_classes_count)
    for cls,prob in zip(res.boxes.cls.cpu().numpy(), res.boxes.conf.cpu().numpy()):
        p[int(cls)] = prob
    return np.array(p)
#-----------------------------------------------------------------------
def predict_yolo_masked(masks,verbose=False):
    imglst_preds = []
    for mask in masks:
        preds = []
        for repl in background_image_set:
            # print(mask.shape, repl.shape)
            if len(mask.shape)!=3:
                mask3 = np.stack([mask,mask,mask], axis=2)
            else:
                print(mask.shape)
                mask3 = mask.copy()
            masked_image = np.where(mask3, image_to_explain, repl)
            preds.append(predict_yolo(masked_image,verbose=verbose))

        preds = np.mean(preds, axis=0)
        imglst_preds.append(preds)       
    
    return np.array(imglst_preds)

In [ ]:
def load_image_to_explain(fname,bg_type='gray', load_gt=True):
    # global predicted_fS, predicted_f0, predicted_cls, sorted_classes, f_S, f_0,sorted_probs
    # global model_type,pretrained_model_type

    image_to_explain,image_to_explain_tensor,background_image_set,background_tensors = load_image(fname,bg_type=bg_type)
    h,w,_ = image_to_explain.shape
    # Foreground image to be explained  
    predicted_fS = predict_yolo(image_to_explain) 
    # predicted_fS = f(torch.unsqueeze(resnet50_preprocess(image_to_explain.astype(np.float32)/255.0).to(device), dim=0))[0]
    sorted_classes = np.flip(np.argsort(predicted_fS))
    sorted_probs   = predicted_fS[sorted_classes]
    predicted_cls = sorted_classes[0]
    f_S = float(predicted_fS[predicted_cls])
    #####################
    
    predicted_f0 = [predict_yolo(bkgnd.astype(np.float32)/255.0) for bkgnd in background_image_set]
    predicted_f0 = np.mean(predicted_f0,axis=0)
    f_0          = float(predicted_f0[predicted_cls])
    return image_to_explain,image_to_explain_tensor,background_image_set,background_tensors,predicted_fS,sorted_classes,sorted_probs,predicted_cls,f_S,predicted_f0,f_0
    # if load_gt:
        # image_no = int(image_path.split('\\')[-1].split('.')[0])

        # load_groundtruth(coco,image_path,fixed_category=fixed_category)
    

## SELECT IMAGE

In [ ]:
## Get available precomputed SAM partitions
# fetch available unique image_ids from the partitions directory
image_ids = os.listdir("../partitions")
image_ids = [f.split('.')[0].split('_')[0] for f in image_ids if '_refined.npy' in f]
print(f'computed image_ids: {len(image_ids)}')

already_computed = ['000000049091',
 '000000000632',
 '000000186929',
 '000000002299',
 '000000225757',
 '000000171382']

image_ids = [img_id for img_id in image_ids if img_id not in already_computed]
print(f'filtered image_ids: {len(image_ids)}')
image_ids

In [ ]:
image_dir = config["data"]["image_dir"] # Update for your image directory
annotation_file =  config["data"]["annotation_file"] # Update for your annotation file
coco = COCO(annotation_file)

categories = coco.loadCats(coco.getCatIds())
coco_categories = {cat['id']: cat['name'] for cat in categories}

In [ ]:
# image_id = '000000171382'  # Example image ID
image_id = '000000170670'  # Example image ID
image_path = os.path.join(image_dir, f'{image_id}.jpg')

In [ ]:
image_to_explain,image_to_explain_tensor,background_image_set,background_tensors,\
    predicted_fS,sorted_classes,sorted_probs,predicted_cls,f_S,predicted_f0,f_0 = load_image_to_explain(image_path, bg_type='gray')
# fixed_category = 'tv'
fixed_category = class_names[predicted_cls]
print('fixed_category: ',fixed_category)

In [ ]:
plt.imshow(image_to_explain)
plt.xticks([]); plt.yticks([]); 
# plt.title(f'Predicted Class: {fixed_category}'); plt.show()


In [ ]:
# image_no = 113235
image_no = int(image_id)

image_info = coco.loadImgs(image_no)[0]
ann_ids = coco.getAnnIds(imgIds=image_info['id'])
annotations = coco.loadAnns(ann_ids)
has_segmentation = any('segmentation' in ann for ann in annotations)
print(f"Segmentation annotations present: {has_segmentation}")

In [ ]:
# image_info

In [ ]:
# categories = coco.loadCats(coco.getCatIds())
# coco_categories = {cat['id']: cat['name'] for cat in categories}
# Get annotations for the selected image
ann_ids = coco.getAnnIds(imgIds=image_info['id'])
annotations = coco.loadAnns(ann_ids)

In [ ]:
# Check if segmentation annotations are present
has_segmentation = any('segmentation' in ann for ann in annotations)
print(f"Segmentation annotations present: {has_segmentation}")
if has_segmentation:
    for ann in annotations:
        if 'segmentation' in ann:
            print(f"Segmentation Annotation: {ann['segmentation']}")
            break

In [ ]:
def get_annotation(coco,image_no,category_name=None):
    if isinstance(image_no, str):
        image_no = int(image_no.split('\\')[-1].split('.')[0])
    
    image_info = coco.loadImgs(image_no)[0]
    if category_name is None:
        annotation_ids = coco.getAnnIds(imgIds=image_info['id'])
    else:
        category_ids = coco.getCatIds(catNms=[category_name])
        annotation_ids = coco.getAnnIds(imgIds=image_info['id'], catIds=category_ids)
    annotations = coco.loadAnns(annotation_ids)
    return annotations

def create_gt(coco,image_no,category_name=None, verbose=False):
    annotations = get_annotation(coco,image_no,category_name=category_name)
    if verbose:
        if len(annotations)>0:
            print(f"Image:{image_info['id']} has {len(annotations)} annotations")
    
    mask = np.zeros((image_info['height'], image_info['width']), dtype=np.uint8)
    category_mask = np.zeros((image_info['height'], image_info['width']), dtype=np.uint8)

    # Combine all masks for this image
    for ann in annotations:
        if 'segmentation' in ann:
            category_id = ann['category_id']  # Unique ID for object category
            # Decode the segmentation mask
            if isinstance(ann['segmentation'], list):  # Polygon format
                for seg in ann['segmentation']:
                    pts = np.array(seg).reshape(-1, 2).astype(np.int32)
                    cv2.fillPoly(mask, [pts], color=1)  # Fill the mask polygon
                    cv2.fillPoly(category_mask, [pts], color=category_id)
            elif isinstance(ann['segmentation'], dict):  # RLE format
                rle = ann['segmentation']
                decoded_mask = coco.annToMask(ann)
                mask += decoded_mask  # Add binary mask
                category_mask[decoded_mask > 0] = category_id  # Assign category ID
    
    # Resize masks to match actual image dimensions
    if mask.shape[:2] != image_to_explain.shape[:2]:
        # print(f"Resizing masks: Annotated={mask.shape}, Actual={image_to_explain.shape[:2]}")
        mask = cv2.resize(mask, (image_to_explain.shape[1], image_to_explain.shape[0]), interpolation=cv2.INTER_NEAREST)
        category_mask = cv2.resize(category_mask, (image_to_explain.shape[1], image_to_explain.shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask,category_mask,annotations

In [ ]:
def load_groundtruth(coco,image_no,fixed_category=None):
    global ground_truth,weighted_ground_truth,annotations
    mask,ground_truth,annotations = create_gt(coco,image_no,category_name = fixed_category)
    weighted_ground_truth = gaussian_filter(ground_truth.astype(float), 16) * ground_truth
    ground_truth.dtype = 'bool'
fixed_category = fixed_category
load_groundtruth(coco,image_no,fixed_category=fixed_category)

In [ ]:
plt.imshow(ground_truth, cmap='gray'); plt.xticks([]); plt.yticks([]); plt.title(f'Ground Truth Mask for {fixed_category}'); plt.show()

In [ ]:
# Perform inference on an image
def plot_predictions(image, results, category_name=None, filter_preds=True,
                     line_thickness=2, exp_type='demo', save_fig=False, fig_size=(3, 3),
                     title=None, selected_ext='png', destroy_fig=False):
    image_ = image.copy()
    plt.figure(figsize=fig_size)
    plt.imshow(image_); plt.axis('off')
    for result in results:
        # Access detected classes, confidences, and boxes
        class_ids = result.boxes.cls.cpu().numpy()  # Class IDs
        scores = result.boxes.conf.cpu().numpy()   # Confidence scores
        boxes = result.boxes.xyxy.cpu().numpy()    # Bounding boxes in xyxy format
        labels = model.names                       # Class labels (MS COCO classes)
        # Draw bounding boxes and labels on the image
        for box, class_id, score in zip(boxes, class_ids, scores):
            label = labels[int(class_id)]
            confidence = f"{score:.2f}"
            x1, y1, x2, y2 = map(int, box)  # Bounding box coordinates
            if filter_preds and category_name and label != category_name:
                continue
            width,height = x2 - x1, y2 - y1
            plt.gca().add_patch(plt.Rectangle((x1, y1), width, height, edgecolor='darkred', facecolor='none', linewidth=line_thickness))
            plt.text(x1, y1 - 5, f"{label} {confidence}", color='white', fontsize=12, bbox=dict(facecolor='darkred', alpha=0.5))
    plt.show()


In [ ]:
results = model.predict(image_to_explain,verbose=True)

In [ ]:
# Extract top-k classes
def get_top_k_classes(results, k=5):
    # Predictions contain bounding boxes with associated scores and classes
    class_names = model.names
    top_k = []
    for result_ in results:
        boxes = result_.boxes  # List of bounding boxes
        for box in boxes:
            score,class_id = box.conf, box.cls  # Confidence score, # Class ID
            top_k.append((int(class_id.item()),class_names[int(class_id.item())], score.item()))
        # Sort by confidence and take top-k
        top_k = sorted(top_k, key=lambda x: x[2], reverse=True)[:k]
    return top_k

In [ ]:
top_classes_n = 5

top_k_classes = get_top_k_classes(results, k=top_classes_n)
print("Top-k Classes (Class ID, Confidence):", top_k_classes)
print([class_names[int(cls)] for cls, _,_ in top_k_classes])
print('-'*120)
print('top_k_classes \t', top_k_classes)

In [ ]:
fixed_category

In [ ]:
plot_predictions(image_to_explain,results,category_name = fixed_category,fig_size=(5,5), save_fig=False)

In [ ]:
# MASKING FUNCTION
def predict_yolo_masked(masks):
    imglst_preds = []
    for mask in masks:
        preds = []
        for repl in background_image_set:
            if len(mask.shape)==2:
                mask3 = np.stack([mask,mask,mask], axis=2)
            else:
                mask3 = mask.copy()
            masked_image = np.where(mask3, image_to_explain, repl)
            preds.append(predict_yolo(masked_image))
        preds = np.mean(preds, axis=0)
        imglst_preds.append(preds)       
    return np.array(imglst_preds)

def predict_yolo(x,verbose=False):
    x = torch.from_numpy(x).to(device)
    x = x.cpu().numpy()
    res = model.predict(x,verbose=verbose)[0]
    p = np.zeros(80)
    for cls,prob in zip(res.boxes.cls.cpu().numpy(), res.boxes.conf.cpu().numpy()):
        p[int(cls)] = prob
    torch.cuda.empty_cache()
    return np.array(p)

In [ ]:
print(image_to_explain.shape)
bg_ls = np.zeros((100,image_to_explain.shape[0],image_to_explain.shape[1],image_to_explain.shape[2]))
# bg_ls = np.random.randint(0, 255, (50,426, 640, 3), dtype=np.uint8)
print(bg_ls.shape)
pred = predict_yolo_masked(bg_ls)
print(pred.shape)
torch.cuda.empty_cache()

In [ ]:
num_explained_classes        = 4
MAX_EVALS_BUDGET             = 100
explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)

In [ ]:
shap_values_bpt = explainer.explain_instance(MAX_EVALS_BUDGET, method='BPT',batch_size=16)
shap_values_aa = explainer.explain_instance(MAX_EVALS_BUDGET, method='AA',  batch_size=16)

In [ ]:
shap_bpt.plot_owen_values(explainer, [shap_values_bpt],class_names, names=['BPT'])

In [ ]:
## PLOT OWEN VALUES
# shap_bpt.plot_owen_values(explainer, [shap_values_aa],class_names, names=['AxisAligned'])

shap_bpt.plot_owen_values(explainer, [shap_values_aa,shap_values_bpt],class_names, names=['AxisAligned','BPT'])

In [ ]:
# Save all detection results for each image to a JSON file used by the HTML report.
import json

def to_jsonable(value):
    if hasattr(value, 'item'):
        return value.item()
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    return value

if isinstance(results, dict):
    yolo_speed = results.get('speed', {})
elif isinstance(results, (list, tuple)) and len(results) > 0:
    yolo_speed = getattr(results[0], 'speed', {})
else:
    yolo_speed = getattr(results, 'speed', {})

detection_summary = {
    'image_id': str(image_no),
    'image_id_padded': str(image_id),
    'input_image_path': image_path,
    'sam_mask_path': str(project_root / 'examples' / 'partitions' / f'{image_id}_sam.png'),
    'fixed_category': fixed_category,
    'explained_class': fixed_category,
    'predicted_class_id': int(predicted_cls),
    'f_S': float(f_S),
    'f_0': float(f_0),
    'has_segmentation': bool(has_segmentation),
    'speed': to_jsonable(yolo_speed),
    'top_k_classes': [
        {'class_id': int(class_id), 'class_name': str(class_name), 'confidence': float(confidence)}
        for class_id, class_name, confidence in top_k_classes
    ],
}

path_results = os.path.join(original_working_dir, 'results')
path_results_img = os.path.join(path_results, str(image_no))
os.makedirs(path_results_img, exist_ok=True)

detection_summary_path = os.path.join(path_results_img, f'{image_no}_results.json')
with open(detection_summary_path, 'w') as f:
    json.dump(to_jsonable(detection_summary), f, indent=2)

print(f'Saved detection summary: {detection_summary_path}')


## Distance Functions

### ShapBPT with Custom BPT

In [ ]:
# explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)
# bptree = shap_bpt.build_bpt_from_image(image_to_explain, use_perim_term=True, use_area_term=True, use_color_term=True)
# print(bptree.N)
# # Compute Owen values with the BPT method
# shap_values_bpt = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =bptree, method='BPT',batch_size=16)
# shap_bpt.plot_owen_values(explainer, shap_values_bpt, class_names)

In [ ]:
# # Create the shap_bpt explainer using our masking-based black-box function
# MAX_EVALS_BUDGET = 100
# explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)
# for use_perim_term in [True, False]:
#     for use_area_term in [True, False]:
#         for use_color_term in [True, False]:
#             bptree = shap_bpt.build_bpt_from_image(image_to_explain, use_perim_term=use_perim_term, use_area_term=use_area_term, use_color_term=use_color_term, 
#                                                 #    prebuilt_partitions=sam_partitions
#                                                 )
#             # Compute Owen values with the BPT method
#             shap_values_bpt = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =bptree, method='BPT',batch_size=16)

#             active_terms = []
#             if use_perim_term: active_terms.append('Perimeter')
#             if use_area_term: active_terms.append('Area')
#             if use_color_term: active_terms.append('Color')
#             string_names = ' | '.join(active_terms) or 'No terms'

#             shap_bpt.plot_owen_values(explainer, shap_values_bpt, class_names, names=[string_names])

In [ ]:
print('Expected Shapley explanation: ', explainer.base_f_S[0] - explainer.base_f_0[0])
print('Computed Shapley explanation: ', np.sum(shap_values_bpt[0]))

## Genearte/LOAD SAM

In [ ]:
partition = cv2.imread(f'../partitions/{image_id}_sam.png',
                    #    cv2.IMREAD_GRAYSCALE
                       )
print(partition.shape)  # (H, W)
plt.imshow(partition, cmap="gray")
plt.axis("off")
plt.show()

## Refined Masks

In [ ]:
masks = np.load(f"../partitions/{image_id}_refined.npy").astype(np.uint16)

print(masks.shape)  # (H, W)
print(len(np.unique(masks)))
im = plt.imshow(masks, cmap="tab20")
plt.colorbar(im, fraction=0.046, pad=0.04, aspect=13.5)
plt.title(f'{len(np.unique(masks))}');
plt.xticks([]); plt.yticks([]);
plt.show()

## WITH SAM

In [ ]:
partitions = masks

print(partitions.shape)
print(partitions.dtype)
print(partitions.min(), partitions.max())
print(np.unique(partitions).size)
print(np.unique(partitions)[:20])

### ONLY SAM

In [ ]:
from scipy import ndimage as ndi
import numpy as np

def cap_partition_labels(partitions, max_labels=64):
    partitions = np.asarray(partitions).astype(np.int64)

    labels, counts = np.unique(partitions, return_counts=True)

    if len(labels) <= max_labels:
        keep_labels = labels
    else:
        # Keep the largest regions, merge tiny extras into nearest kept region
        keep_labels = labels[np.argsort(counts)[-max_labels:]]

    keep_mask = np.isin(partitions, keep_labels)

    if not np.all(keep_mask):
        # For every removed pixel, copy nearest kept label
        _, nearest_idx = ndi.distance_transform_edt(~keep_mask, return_indices=True)
        partitions = partitions.copy()
        partitions[~keep_mask] = partitions[tuple(idx[~keep_mask] for idx in nearest_idx)]

    # Remap labels to contiguous 0..K-1
    unique_ids = np.unique(partitions)
    remap = {old: new for new, old in enumerate(unique_ids)}
    partitions = np.vectorize(remap.get)(partitions).astype(np.uint8)

    return partitions

In [ ]:
path_results = os.path.join(original_working_dir, 'results')
path_results_img = os.path.join(path_results, str(image_no))
os.makedirs(path_results, exist_ok=True)
os.makedirs(path_results_img, exist_ok=True)


In [ ]:

from matplotlib.colors import LinearSegmentedColormap

# Custom colormap for Shapley values - similar to 'seismic' but with lighter tones.
shapley_values_colormap = LinearSegmentedColormap.from_list("shapley_values_colormap", 
                                                            [(0.0, '#0053d1'),
                                                             (0.2, '#248df4'),
                                                             (0.5, 'white'),  
                                                             (0.8, '#f23754'),
                                                             (1.0, '#cb0021')])


def plot_owen_values(explainer, shap_values, class_names,
                     figure_name, save_plot=True,savepath=None, names=None,
                     destroy_fig=False):
    """
    Visualize ShapBPT explanations.

    Parameters
    ----------
    explainer : Explainer
        Fitted explanation object.

    shap_values : np.ndarray
        Explanation maps.

    class_names : list[str]
        Names of model output classes.

    names : list[str], optional
        Row labels for multiple explanation sets.

    Returns
    -------
    None
        Displays a matplotlib figure.
    """
    shap_values = np.array(shap_values)
    if len(shap_values.shape)==3: shap_values = np.array([shap_values])
    max_val = np.nanpercentile(np.abs(shap_values.flatten()), 99.9)
    num_explained_classes = len(explainer.base_f_S)
    num_rows = len(shap_values)
    fig,axes = plt.subplots(num_rows+1, num_explained_classes+1, 
                            figsize=(2*(num_explained_classes+1), 2*(num_rows+0.3)), 
                            squeeze=False,
                            height_ratios=[1]*num_rows + [0.3])
    base_image = explainer.image_to_explain
    if np.max(base_image)>1: base_image = base_image.astype(np.uint8)
    if len(base_image.shape)==2:
        base_image = np.stack([base_image, base_image, base_image], axis=-1)
    img_grey = (0.2989 * base_image[:, :, 0] +
                0.5870 * base_image[:, :, 1] + 
                0.1140 * base_image[:, :, 2])
    # axes[0].set_title(f'real: {class_names[expected_class]}')
    for r in range(num_rows):
        axes[r,0].imshow(base_image)
        for i in range(num_explained_classes):
            axes[r,i+1].imshow(img_grey.astype(base_image.dtype), alpha=0.50, cmap='gray')
            im=axes[r,i+1].imshow(shap_values[r,i], cmap=shapley_values_colormap, vmin = -max_val, vmax = max_val, alpha=0.80)
            if r==0: axes[r,i+1].set_title(f'{class_names[explainer.output_indexes[i]]}', fontsize=10)#+
                                #f'\n{explainer.base_f_S[i]:.5} to {explainer.base_f_0[i]:.5}')
        for jjj in range(num_explained_classes+1): axes[r,jjj].set_xticks([]) ; axes[r,jjj].set_yticks([])
    if names is not None:
        for r in range(num_rows):
            axes[r,0].set_ylabel(names[r])
    # Use the last row for the colorbar
    for ax in axes[-1,:]:
        ax.set_axis_off()
        # ax.set_box_aspect(0.1)
    cb = fig.colorbar(im, ax=axes[-1,:], label="Shapley/Owen value", 
                      orientation="horizontal", aspect=80, fraction=0.9)#, location='bottom') #,  fraction=0.5, 
    cb.outline.set_visible(False)
    fig.subplots_adjust(hspace=0.1, wspace=0.1)
    if save_plot: plt.savefig(os.path.join(savepath, figure_name), dpi=300, bbox_inches='tight')
    # plt.tight_layout()
    if destroy_fig: 
        plt.close(fig)  
    plt.show()


def plot_single_attributions(shap_values, savepath, figure_name, save_plot=True, factor=1,
                             destroy_fig=True, robust_percentile=99.5):
    exp_code, _ = figure_name.split('_')[:2]

    shap_values = np.array(shap_values)
    shap_values_1 = shap_values[0] if shap_values.ndim == 3 else shap_values
    if shap_values_1.ndim != 2:
        raise ValueError(f'Expected a 2D attribution map or a 3D class stack, got {shap_values.shape}')

    finite_abs = np.abs(shap_values_1[np.isfinite(shap_values_1)])
    nonzero_abs = finite_abs[finite_abs > 0]
    if nonzero_abs.size == 0:
        max_val = 1.0
    else:
        max_val = np.nanpercentile(nonzero_abs, robust_percentile)
        if not np.isfinite(max_val) or max_val <= 0:
            max_val = np.nanmax(nonzero_abs)
        max_val = max_val + factor * 1e-12

    # print(f'{figure_name}: color scale [-{max_val:.4g}, {max_val:.4g}]')
    fig = plt.figure(figsize=(6, 6))
    ax = plt.imshow(shap_values_1, cmap=shapley_values_colormap, vmin = -max_val, vmax = max_val)
    plt.colorbar(ax, fraction=0.03)#, location='bottom') #,  fraction=0.5, 
    plt.xticks([]); plt.yticks([]);
    # plt.tight_layout()
    if save_plot: plt.savefig(os.path.join(savepath, figure_name), dpi=300, bbox_inches='tight')
    if destroy_fig: plt.close(fig)  
    plt.show()

# plot_single_attributions(shap_values_bpt_sam, path_results,  figure_name)

## CREATE A MAPPING TABLE for this


In [ ]:
import pandas as pd

experiment_map = {
    "E1": {"use_area_term": True,  "use_perim_term": True,  "use_color_term": True},
    "E2": {"use_area_term": True,  "use_perim_term": True,  "use_color_term": False},
    "E3": {"use_area_term": True,  "use_perim_term": False, "use_color_term": True},
    "E4": {"use_area_term": True,  "use_perim_term": False, "use_color_term": False},
    "E5": {"use_area_term": False, "use_perim_term": True,  "use_color_term": True},
    "E6": {"use_area_term": False, "use_perim_term": True,  "use_color_term": False},
    "E7": {"use_area_term": False, "use_perim_term": False, "use_color_term": True},
    "E8": {"use_area_term": False, "use_perim_term": False, "use_color_term": False},
}

# Direct lookup table (E1 -> flags)
experiment_table = pd.DataFrame.from_dict(experiment_map, orient="index")
experiment_table.index.name = "Exp"
experiment_table = experiment_table.reset_index()
experiment_table["description"] = experiment_table.apply(
    lambda row: "+".join(
        part for part, enabled in [
            ("area", row["use_area_term"]),
            ("perimeter", row["use_perim_term"]),
            ("color", row["use_color_term"]),
        ]
        if enabled
    ) or "none",
    axis=1,
)

# Inverse lookup table (flags -> E1)
experiment_inverse_map = {
    (cfg["use_area_term"], cfg["use_perim_term"], cfg["use_color_term"]): exp
    for exp, cfg in experiment_map.items()
}

print(experiment_table)

# Save this table if needed:
# experiment_table.to_csv("experiment_mapping_table.csv", index=False)
# with open("experiment_mapping_table.json", "w") as f:
#     experiment_table.to_json(f, orient="records", indent=2)


def get_experiment_config(code: str):
    """Return the configuration dictionary for an experiment code like E1."""
    return experiment_map[code]


def get_experiment_code(use_area_term: bool, use_perim_term: bool, use_color_term: bool):
    """Return the experiment code like E1 for a given flag combination."""
    key = (use_area_term, use_perim_term, use_color_term)
    return experiment_inverse_map[key]


def explain_experiment(code: str):
    """Return a readable description of an experiment code like E1."""
    cfg = get_experiment_config(code)
    active = [name.replace("use_", "").replace("_term", "") for name, enabled in cfg.items() if enabled]
    return f"{code}: {' + '.join(active) if active else 'none'}"


def explain_flags(use_area_term: bool, use_perim_term: bool, use_color_term: bool):
    """Return a readable description for a given flag combination."""
    code = get_experiment_code(use_area_term, use_perim_term, use_color_term)
    return explain_experiment(code)

# Examples:
# print(get_experiment_code(True, True, True))   # E1
# print(get_experiment_config("E1"))
# print(explain_experiment("E1"))
# print(explain_flags(True, True, True))

In [ ]:
print(get_experiment_code(True, True, True))   # E1
print(get_experiment_config("E1"))
print(explain_experiment("E1"))
print(explain_flags(True, True, True))

In [ ]:
MAX_EVALS_BUDGET = 100

### ONLY SAM

In [ ]:
explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)
partitions = cap_partition_labels(masks, max_labels=63)
bptree = shap_bpt.build_bpt_from_image(
                        image_to_explain,
                        use_perim_term=False,
                        use_area_term=False,
                        use_color_term=False,
                        prebuilt_partitions=partitions,
)
print(bptree.N)
# Compute Owen values with the BPT method
shap_values_sam = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =bptree, method='BPT',batch_size=16)
shap_bpt.plot_owen_values(explainer, shap_values_sam, class_names, names=['SAM'])

## Color + Area + Perimeter + SAM

In [ ]:
explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)
partitions = cap_partition_labels(masks, max_labels=63)
bptree = shap_bpt.build_bpt_from_image(image_to_explain,
                                        use_perim_term=True,
                                          use_area_term=True,
                                            use_color_term=True,
                                            prebuilt_partitions=partitions
                                            )
print(bptree.N)
# Compute Owen values with the BPT method
shap_values_bpt_sam = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =bptree, method='BPT',batch_size=16)
shap_bpt.plot_owen_values(explainer, shap_values_bpt_sam, class_names, names=['BPT+SAM'])

In [ ]:
shap_values_all_4 = []

for exp_code, shap_values in zip(['AA','BPT','SAM','BPT-SAM'],[shap_values_aa, shap_values_bpt, shap_values_sam, shap_values_bpt_sam]):
    data = {
        'exp_code': exp_code,
        'shap_values': shap_values[0]
    }
    shap_values_all_4.append(data)
    print(exp_code, np.max(np.abs(shap_values)))
    plot_single_attributions(shap_values, path_results_img,  f'{exp_code}_{image_no}',robust_percentile=99.9985, destroy_fig=False)

## ALL Distance Functions

In [ ]:
# # Create the shap_bpt explainer using our masking-based black-box function
# explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)
# partitions = cap_partition_labels(masks, max_labels=63)
# from tqdm.auto import tqdm
# # add a progress bar for the nested loops based on all combinations of the four boolean flags (2^4 = 16 combinations)
# combination_count = 2 ** 4 # 2^4 combinations
# pbar = tqdm(total=combination_count)  
# shap_values_all = []
# for use_perim_term in [True, False]:
#     for use_area_term in [True, False]:
#         for use_color_term in [True, False]:
#             for use_sam_partitions in [True, False]:
#                 bptree = shap_bpt.build_bpt_from_image(image_to_explain,
#                                                        use_perim_term=use_perim_term,
#                                                        use_area_term=use_area_term,
#                                                        use_color_term=use_color_term,
#                                                        prebuilt_partitions=partitions if use_sam_partitions else None
#                                                        )
#                 # Compute Owen values with the BPT method
#                 shap_values_bpt_sam = explainer.explain_instance(MAX_EVALS_BUDGET, method='BPT',
#                                                             bpt=bptree,
#                                                             batch_size=64)
#                 active_terms = []
#                 if use_perim_term: active_terms.append('Perimeter')
#                 if use_area_term: active_terms.append('Area')
#                 if use_color_term: active_terms.append('Color')
#                 if use_sam_partitions: active_terms.append('SAM')
#                 string_names = ' | '.join(active_terms) or 'No terms'

#                 exp_code = get_experiment_code(use_area_term, use_perim_term, use_color_term)
#                 figure_name = f'{exp_code}_{"SAM" if use_sam_partitions else "noSAM"}_{image_no}.png'

#                 plot_owen_values(explainer, shap_values_bpt_sam, class_names,figure_name, names=[string_names], save_plot=False, savepath= path_results_img)
#                 plot_single_attributions(shap_values_bpt_sam, path_results_img,  figure_name)
#                 data = {'exp_code':exp_code,
#                  'shap_values':shap_values_bpt_sam[0]
#                  }
#                 shap_values_all.append(data)
#                 pbar.update(1)  # Update the progress bar after each combination
#                 # plt.gcf().savefig(os.path.join(path_results, figure_name), dpi=300, bbox_inches='tight')
#                 # print(f'Saved figure: {figure_name}')
#                 ## save this plot if needed

## HTML REPORT

In [ ]:
import sys

scripts_dir = project_root / "examples/scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import utils as ut

import importlib
importlib.reload(ut)    

## Generate Report for all Images

In [ ]:
ut.build_all_images_html_report(
    path_results,
    os.path.join(path_results, "shapbpt_all_images_report.html")
)

## Evaluation

In [ ]:
predicted_cls = explainer.output_indexes[0]
f_S = explainer.base_f_S[0]
f_0 = explainer.base_f_0[0]

aucD_bpt = ut.saliency_to_auc(predict_yolo_masked, shap_values_bpt_sam[0], f_S, f_0, predicted_cls, method='del')
aucI_bpt = ut.saliency_to_auc(predict_yolo_masked, shap_values_bpt_sam[0], f_S, f_0, predicted_cls, method='ins')

# shap_values_sam
# shap_values_bpt_sam
# shap_values_aa


In [ ]:
def compute_auc_results(shap_values_all, nu, f_S, f_0, predicted_cls, batch_size=4, verbose=True):
    auc_results = []
    label_counts = {}

    for exp_data in shap_values_all:
        exp_code = exp_data['exp_code']
        base_label = exp_data.get('label', exp_code)
        label_counts[base_label] = label_counts.get(base_label, 0) + 1
        label = base_label if label_counts[base_label] == 1 else f"{base_label} #{label_counts[base_label]}"

        shap_values = exp_data['shap_values']
        auc_del = ut.saliency_to_auc(nu, shap_values, f_S, f_0, predicted_cls, batch_size=batch_size, method='del')
        auc_ins = ut.saliency_to_auc(nu, shap_values, f_S, f_0, predicted_cls, batch_size=batch_size, method='ins')

        auc_results.append({
            **exp_data,
            'label': label,
            'auc_del': auc_del,
            'auc_ins': auc_ins,
        })
        if verbose:
            print(f"{label}: AUC-DEL={auc_del['auc_clipr']:.4f}, AUC-INS={auc_ins['auc_clipr']:.4f}")

    return auc_results


def plot_auc_results(auc_results, figsize=(11, 4), fill_alpha=0.08, save_plot=False,
                     image_no=None,
                     destroy_figs=False):
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=True, sharey=True)
    cmap = plt.get_cmap('tab20')

    best_ins = max(r['auc_ins']['auc_clipr'] for r in auc_results)
    best_del = min(r['auc_del']['auc_clipr'] for r in auc_results)

    panels = [
        (axes[0], 'auc_ins', best_ins, True, '$\\mathit{AUC}^{+}$', 'lower right'),
        (axes[1], 'auc_del', best_del, False, '$\\mathit{AUC}^{-}$', 'upper right'),
    ]

    for ax, auc_key, best_auc, higher_is_better, title, legend_loc in panels:
        sorted_results = sorted(
            auc_results,
            key=lambda result: result[auc_key]['auc_clipr'],
            reverse=higher_is_better,
        )

        for idx, result in enumerate(sorted_results):
            auc = result[auc_key]
            score = auc['auc_clipr']
            is_best = np.isclose(score, best_auc)
            color = cmap(idx % cmap.N)
            star = '*' if is_best else ''

            ax.plot(
                auc['xs'],
                auc['y_clipr'],
                color=color,
                lw=2.4 if is_best else 1.4,
                alpha=0.95 if is_best else 0.75,
                label=f"{star}{result['label']} {score:.4f}",
            )
            ax.fill_between(auc['xs'], auc['y_clipr'], color=color, alpha=fill_alpha)

        ax.axhline(1.0, ls='--', c='grey', zorder=0)
        ax.axhline(0.0, c='lightgrey', zorder=0)
        ax.set_title(title, fontsize=16)
        ax.set_xlabel('Fraction of Pixels Removed/Inserted')
        ax.grid(alpha=0.25)
        ax.legend(borderpad=0.2, labelspacing=0.1, loc=legend_loc, fontsize=8)

    axes[0].set_ylabel('Model Confidence')
    plt.tight_layout()
    if save_plot:
        plt.savefig(os.path.join(path_results_img, f'auc_results_{image_no}.png'), dpi=300, bbox_inches='tight')
    if destroy_figs:
        plt.close(fig)
    plt.show()
    return fig, axes


In [ ]:
# auc_results = compute_auc_results(shap_values_bpt_sam, predict_yolo_masked, f_S, f_0, predicted_cls, batch_size=4)
# fig, axes = plot_auc_results(auc_results)

In [ ]:
auc_results = compute_auc_results(shap_values_all_4, predict_yolo_masked, f_S, f_0, predicted_cls, batch_size=4)
fig, axes = plot_auc_results(auc_results)

In [ ]:
report_path = ut.build_html_report(
    path_results_img,
    os.path.join(path_results, f'shapbpt_report_{image_no}.html'),
    experiment_map,
    experiment_table,
    image_no,
)
print(f'Saved HTML report: {image_no}')  

## RUN FULL PIPELINE

In [ ]:
# run_fullpipeline = False
run_fullpipeline = True

In [ ]:
# save_plots = True
save_plots = False

# destroy_figs = False
destroy_figs = True

# verbose = True
verbose = False

if run_fullpipeline:

    num_explained_classes        = 4
    MAX_EVALS_BUDGET             = 100
    from tqdm.auto import tqdm
    # for image_id in ['000000014439', '000000015660', '000000170670', '000000017899']:
    for image_id in tqdm(image_ids):
        print('='*100)
        image_dir = config["data"]["image_dir"] # Update for your image directory
        annotation_file =  config["data"]["annotation_file"] # Update for your annotation file
        # image_id = '000000171382'  # Example image ID
        # image_id = '000000170670'  # Example image ID
        image_path = os.path.join(image_dir, f'{image_id}.jpg')

        #######################################
        image_to_explain,image_to_explain_tensor,\
            background_image_set,background_tensors,\
                predicted_fS,sorted_classes,sorted_probs,\
                    predicted_cls,f_S,predicted_f0,f_0 = load_image_to_explain(image_path,
                                                                               bg_type='gray')
        
        fixed_category = class_names[predicted_cls]
        print('fixed_category: ',fixed_category)

        #######################################
        fig = plt.figure(figsize=(6, 6))
        plt.imshow(image_to_explain)
        plt.xticks([]); plt.yticks([]); 
        if destroy_figs:
            plt.close(fig)
        plt.show()
        #######################################
        image_no = int(image_id)

        image_info = coco.loadImgs(image_no)[0]
        ann_ids = coco.getAnnIds(imgIds=image_info['id'])
        annotations = coco.loadAnns(ann_ids)
        has_segmentation = any('segmentation' in ann for ann in annotations)
        if verbose:
            print(f"Segmentation annotations present: {has_segmentation}")
        #######################################
        # Get annotations for the selected image
        ann_ids = coco.getAnnIds(imgIds=image_info['id'])
        annotations = coco.loadAnns(ann_ids)
        #  LOAD GT
        if load_groundtruth:
            load_groundtruth(coco,image_no,fixed_category=fixed_category)
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(ground_truth, cmap='gray')
            plt.xticks([]); plt.yticks([])
            plt.title(f'Ground Truth Mask for {fixed_category}')
            if destroy_figs:
                plt.close()
            plt.show()
        results = model.predict(image_to_explain,verbose=True)
        
        top_k_classes = get_top_k_classes(results, k=top_classes_n)
        if verbose:
            print("Top-k Classes (Class ID, Confidence):", top_k_classes)
            print([class_names[int(cls)] for cls, _,_ in top_k_classes])
            print('-'*120)
            print('top_k_classes \t', top_k_classes)
        
        # plot predictions for the fixed category
        plot_predictions(image_to_explain,results,
                        category_name = fixed_category,
                        fig_size=(5,5), save_fig=False, destroy_fig=destroy_figs)
        # shapBPT
        explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)

        shap_values_bpt = explainer.explain_instance(MAX_EVALS_BUDGET, method='BPT',batch_size=16)
        shap_values_aa = explainer.explain_instance(MAX_EVALS_BUDGET, method='AA',  batch_size=16)

        # shap_bpt.plot_owen_values(explainer, [shap_values_bpt],class_names, names=['BPT'])
        # shap_bpt.plot_owen_values(explainer, [shap_values_aa,shap_values_bpt],
        #                           class_names, names=['AxisAligned','BPT'])

        path_results_img = os.path.join(path_results, str(image_no))
        os.makedirs(path_results_img, exist_ok=True)

        detection_summary = {
            'image_id': str(image_no),
            'image_id_padded': str(image_id),
            'input_image_path': image_path,
            'sam_mask_path': str(project_root / 'examples' / 'partitions' / f'{image_id}_sam.png'),
            'fixed_category': fixed_category,
            'explained_class': fixed_category,
            'predicted_class_id': int(predicted_cls),
            'f_S': float(f_S),
            'f_0': float(f_0),
            'has_segmentation': bool(has_segmentation),
            'speed': to_jsonable(yolo_speed),
            'top_k_classes': [
                {'class_id': int(class_id), 'class_name': str(class_name), 'confidence': float(confidence)}
                for class_id, class_name, confidence in top_k_classes
            ],
        }

        path_results = os.path.join(original_working_dir, 'results')
        path_results_img = os.path.join(path_results, str(image_no))
        os.makedirs(path_results_img, exist_ok=True)

        detection_summary_path = os.path.join(path_results_img, f'{image_no}_results.json')
        with open(detection_summary_path, 'w') as f:
            json.dump(to_jsonable(detection_summary), f, indent=2)
        if verbose:
            print(f'Saved detection summary: {detection_summary_path}')
            print('Expected Shapley explanation: ', explainer.base_f_S[0] - explainer.base_f_0[0])
            print('Computed Shapley explanation: ', np.sum(shap_values_bpt[0]))

        ## LOAD SAM PARTITIONS
        partition = cv2.imread(f'../partitions/{image_id}_sam.png')
        
        fig = plt.figure(figsize=(6, 6))
        plt.imshow(partition, cmap="gray")
        plt.axis("off")
        if destroy_figs:
            plt.close(fig)
        plt.show()

        ## LOAD REFINED PARTITIONS
        masks = np.load(f"../partitions/{image_id}_refined.npy").astype(np.uint16)
        if verbose:
            print(len(np.unique(masks)))
        fig = plt.figure(figsize=(6, 6))
        im = plt.imshow(masks, cmap="tab20")
        plt.colorbar(im, fraction=0.046, pad=0.04, aspect=13.5)
        plt.title(f'{len(np.unique(masks))}');
        plt.xticks([]); plt.yticks([]);
        if destroy_figs:
            plt.close(fig)
        plt.show()

        if verbose:
            print(get_experiment_code(True, True, True))   # E1
            print(get_experiment_config("E1"))
            print(explain_experiment("E1"))
            print(explain_flags(True, True, True))

        # ONLY SAM
        # explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain,
        #                                  num_explained_classes=num_explained_classes,
        #                                  verbose=True)
        partitions = cap_partition_labels(masks, max_labels=63)
        bptree = shap_bpt.build_bpt_from_image(image_to_explain,
                                               use_perim_term=False,
                                               use_area_term=False,
                                               use_color_term=False,
                                               prebuilt_partitions=partitions,
                                               )
        if verbose: print(bptree.N)
        # Compute Owen values with the BPT method
        shap_values_sam = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =bptree, method='BPT',batch_size=16)
        # plot_owen_values(explainer, shap_values_sam, class_names, names=['SAM'],
        #                  'SAM',
        #                  destroy_fig=destroy_figs)

        ## BPT + SAM
        # explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain,
                                        #  num_explained_classes=num_explained_classes, verbose=True)
        partitions = cap_partition_labels(masks, max_labels=63)
        bptree = shap_bpt.build_bpt_from_image(image_to_explain,
                                                use_perim_term=True,
                                                use_area_term=True,
                                                    use_color_term=True,
                                                    prebuilt_partitions=partitions
                                                    )
        if verbose: print(bptree.N)
        # Compute Owen values with the BPT method
        shap_values_bpt_sam = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =bptree, method='BPT',batch_size=16)
        # plot_owen_values(explainer, shap_values_bpt_sam, class_names, names=['BPT+SAM'],
        #                  destroy_fig=destroy_figs)

        ## SAVE HEATMAPS
        shap_values_all_4 = []

        for exp_code, shap_values in zip(['AA','BPT','SAM','BPT-SAM'],[shap_values_aa, shap_values_bpt, shap_values_sam, shap_values_bpt_sam]):
            data = {'exp_code': exp_code,
                    'shap_values': shap_values[0]
                    }
            shap_values_all_4.append(data)
            if verbose: print(exp_code, np.max(np.abs(shap_values)))
            plot_single_attributions(shap_values, path_results_img,
                                     f'{exp_code}_{image_no}',robust_percentile=99.9985,
                                     save_plot=save_plots,
                                     destroy_fig=destroy_figs)
        
        ## ALL COMBINATIONS OF 4 FLAGS
        # Create the shap_bpt explainer using our masking-based black-box function
        # explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain,
                                        #  num_explained_classes=num_explained_classes, verbose=True)
        partitions = cap_partition_labels(masks, max_labels=63)
        
        # add a progress bar for the nested loops based on all combinations of the four boolean flags (2^4 = 16 combinations)
        combination_count = 2 ** 4 # 2^4 combinations
        pbar = tqdm(total=combination_count)  
        shap_values_all = []
        for use_perim_term in [True, False]:
            for use_area_term in [True, False]:
                for use_color_term in [True, False]:
                    for use_sam_partitions in [True, False]:
                        bptree = shap_bpt.build_bpt_from_image(image_to_explain,
                                                            use_perim_term=use_perim_term,
                                                            use_area_term=use_area_term,
                                                            use_color_term=use_color_term,
                                                            prebuilt_partitions=partitions if use_sam_partitions else None
                                                            )
                        # Compute Owen values with the BPT method
                        shap_values_bpt_sam = explainer.explain_instance(MAX_EVALS_BUDGET, method='BPT',
                                                                    bpt=bptree,
                                                                    batch_size=64)
                        active_terms = []
                        if use_perim_term: active_terms.append('Perimeter')
                        if use_area_term: active_terms.append('Area')
                        if use_color_term: active_terms.append('Color')
                        if use_sam_partitions: active_terms.append('SAM')
                        string_names = ' | '.join(active_terms) or 'No terms'

                        exp_code = get_experiment_code(use_area_term, use_perim_term, use_color_term)
                        figure_name = f'{exp_code}_{"SAM" if use_sam_partitions else "noSAM"}_{image_no}.png'

                        plot_owen_values(explainer, shap_values_bpt_sam, class_names,figure_name,
                                         names=[string_names], save_plot=save_plots,
                                         savepath= path_results_img,
                                         destroy_fig=destroy_figs)
                        plot_single_attributions(shap_values_bpt_sam,
                                                 path_results_img,
                                                 figure_name,
                                                 save_plots,
                                                 destroy_fig=destroy_figs)
                        data = {'exp_code':exp_code,
                        'shap_values':shap_values_bpt_sam[0]
                        }
                        shap_values_all.append(data)
                        pbar.update(1)  # Update the progress bar after each combination
        
        ## COMPUTE AUC RESULTS
        auc_results = compute_auc_results(shap_values_all_4, predict_yolo_masked, f_S, f_0,
                                           predicted_cls, batch_size=4, verbose=verbose)
        fig, axes = plot_auc_results(auc_results, save_plot=save_plots, image_no=image_no,
                                     destroy_figs=destroy_figs)
        
        report_path = ut.build_html_report(path_results_img,
                                           os.path.join(path_results,
                                                        f'shapbpt_report_{image_no}.html'),
                                            experiment_map,
                                            experiment_table,
                                            image_no,
                                            )
        if verbose: print(f'Saved HTML report: {image_no}') 

In [ ]:
report_path = ut.build_html_report(path_results_img,
                                           os.path.join(path_results,
                                                        f'shapbpt_report_{image_no}.html'),
                                            experiment_map,
                                            experiment_table,
                                            image_no,
                                            )

In [ ]:
# fig, axes = plot_auc_results(auc_results, save_plot=save_plots, image_no=image_no)


In [ ]:
# ut.build_all_images_html_report(
#     path_results,
#     os.path.join(path_results, "shapbpt_all_images_report.html")
# )

## END